# Publication-ready figures — Non-uniformity manuscript

Clean notebook for publication-ready code and figures. Imports, data loading, and the **analysis functions** (`median_and_iqr_errors`, `WD_quick`, `process_run_data`, `circle_WD`) are identical to `Nonuniformity_manuscript.ipynb`. Changes: (1) a fixed random seed, (2) every function defined once at the top, (3) all plotting style centralised to match the Figure 1 WD-vs-time panel — including lifting the embedded style out of `plot_multiple_errorbars_iqr`, and (4) a flag-driven config for which Arp2/3 distributions appear in the Figure 1 WD-vs-time-across-distributions panel, with Wasserstein distances expressed in radians.


## Setup & imports

In [ ]:
import pickle
import sys
# from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import pandas as pd
from matplotlib.ticker import MaxNLocator
import numpy as np
import os
from scipy.spatial import procrustes
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from statannotations.Annotator import Annotator

import matplotlib.pyplot as plt
import scipy
from scipy.interpolate import UnivariateSpline
from scipy.spatial import procrustes
from scipy.stats import bootstrap, wasserstein_distance, iqr
import matplotlib.pylab as pl
import ot
from scipy.special import iv
try:
    import pycircular as pc
    has_pycircular = True
except ImportError:
    has_pycircular = False
import warnings
    
# how to refer to these functions in a different folder? Need to compile as a package?
# from cytosim_plot_functions import *
# from cytosim_output_functions import *
import matplotlib.cm as cm
save_figures = True

# working_dir = '/Users/makamats/Google Drive/drubin_lab/Modeling/cytosim_current/cytosim/parameter_sweeps_maxReporting/simulations_cortex/'

# default working directory is now the cytosim/folder. From when you open this notebook.
# working_dir = os.path.abspath(os.path.join(os.curdir, '..'))
# this sets the working directory as the folder above the folder where this notebook appears.
# working_dir = os.path.abspath('./..')

# or:
# working_dir =  '/Users/makamats/Google Drive/drubin_lab/Modeling/cytosim_dmitrieff/cytosim/'
# output_dir = working_dir + '/simulations/endocytosis_mammalian_timestep_vary_sameseed_output'

working_dir = '/Volumes/homes/akamatsuadmin'
simulations_dir = '/_Cytosim/abhi_simulations/'
#output_name = 'baseline_sims'
output_name = 'endocytosis2_baseline_aws'

output_dir = working_dir + simulations_dir + output_name + '_output'
output_dir


In [ ]:
working_dir


In [ ]:
# Secondary imports + run-prefix string.
# NOTE vs parent: the STYLE lines from the parent's cell 2 -- plt.style.use(),
# plt.cool(), sns.set_style('whitegrid'), and the SMALL/MEDIUM/BIGGER plt.rc(...)
# font block -- have been REMOVED. All styling now lives in the centralised
# style cell above. The imports and the datetime/pref logic are unchanged.
%matplotlib inline

import seaborn as sns  # nicer plotting
from scipy.stats import binned_statistic_2d
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import datetime

now = datetime.datetime.now()
date = now.strftime('%Y%m%d')
pref = date+'_'+output_name
pref


## Reproducibility

In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility: fix the global NumPy random seed.
#
# The Wasserstein-distance functions (WD_quick, circle_WD) draw their uniform
# reference / baseline samples from the *legacy* global generator np.random.
# Seeding it once here makes a top-to-bottom run fully deterministic, so the
# WD values and IQR bands stop drifting between runs.
#
# We use np.random.seed (NOT default_rng) on purpose: the analysis functions
# call np.random.uniform on the global generator and are kept identical to the
# parent. A Generator object would require editing those functions.
#
# NOTE: because the functions read the *global* RNG state, full reproducibility
# means running in order from this cell (Restart & Run All), or re-running this
# cell before the compute cells. For a given Figure 2 configuration, a fresh
# top-to-bottom run reproduces the same numbers exactly.
# ---------------------------------------------------------------------------
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)


## Plotting style (centralised — matches the Fig 1 WD-vs-time panel)

In [ ]:
# ---------------------------------------------------------------------------
# Centralised plotting style.
#
# These rcParams define the look of the Figure 1 "Wasserstein distance vs
# time" panels and apply GLOBALLY, so individual plotting cells (and the
# plotting helper) no longer set fonts / ticks / grids / spines. Sizes follow
# the Biophysical Journal / Cell Press spec: Arial (ticks/body 6 pt, axis labels 8 pt bold), 0.6 pt axis/tick
# lines, labelpad 1.0 (previously 12 pt / 1.0 pt line / labelpad 5.0).
#
# Original per-call styling  ->  rcParams equivalent
#   set_x/ylabel(fontsize=12, fontweight='bold', fontfamily='Arial',
#                labelpad=5.0)  -> font.family / axes.labelsize /
#                                  axes.labelweight / axes.labelpad
#   grid(False)                 -> axes.grid = False
#   tick_params(direction='out', length=6, width=1, colors='black',
#               bottom=True, left=True)
#                               -> xtick/ytick.direction / major.size /
#                                  major.width / color + bottom/left
#   spines['top'/'right'].set_visible(False)
#                               -> axes.spines.top / axes.spines.right = False
# ---------------------------------------------------------------------------
import matplotlib as mpl

mpl.rcParams.update({
    # --- Fonts: Arial; body/ticks 6 pt; AXIS LABELS 8 pt BOLD; labelpad 1 (Cell Press / Biophysical J.) ---
    'font.family':      'Arial',
    'font.size':        6,
    'axes.labelsize':   8,
    'axes.labelweight': 'bold',
    'axes.labelpad':    1.0,
    'axes.titlesize':   7,
    'xtick.labelsize':  6,
    'ytick.labelsize':  6,
    'legend.fontsize':  6,

    # --- No grid ---
    'axes.grid':        False,

    # --- Spines / axes lines: hide top & right, 0.6 pt weight ---
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.edgecolor':    'black',
    'axes.linewidth':    0.6,

    # --- Ticks: outward, length 3, width 0.6, black, bottom/left only ---
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'xtick.major.size':  3,
    'ytick.major.size':  3,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.color':       'black',
    'ytick.color':       'black',
    'xtick.bottom':      True,
    'ytick.left':        True,

    # --- Keep text editable as real fonts in vector exports (Inkscape / Illustrator) ---
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

# Confirm Arial is actually available -- matplotlib falls back SILENTLY to
# DejaVu Sans otherwise, so a figure can look fine while not being Arial.
from matplotlib import font_manager
try:
    _arial_path = font_manager.findfont('Arial', fallback_to_default=False)
    print(f'Arial resolved to: {_arial_path}')
except Exception:
    print('WARNING: Arial not found -- matplotlib will fall back to its default '
          'sans-serif font. Install Arial or register the .ttf via '
          'font_manager.fontManager.addfont(path).')


## Output paths

In [ ]:
# ---------------------------------------------------------------------------
# Output paths -- single source of truth for figure saving.
#
# base_savedir previously lived only in the Figure 1 cell, so the save lines
# raised a NameError if that cell hadn't been run. Defining it once here lets
# every figure cell reference the same location. The per-figure subfolders are
# created up front (guarded) so savefig() won't fail on a missing directory.
# The save calls themselves are unchanged and stay commented until you want them.
# ---------------------------------------------------------------------------
import os

base_savedir = '/Volumes/homes/akamatsuadmin/_Cytosim/abhi_simulations/figures/nonuniformity_manuscript'

for _sub in ('figure1', 'figure2', 'figure3', 'figure4'):
    try:
        os.makedirs(os.path.join(base_savedir, _sub), exist_ok=True)
    except OSError:
        pass  # e.g. network volume not mounted -- ignore until you actually save



def save_panel(fig, name, subdir='figure1', w_mm=None, h_mm=None):
    """Export a panel at its FINAL physical size, as vector, for 1:1 Inkscape placement.

    The golden rule for font-size compliance: size the panel HERE (matplotlib),
    then drop the SVG into Inkscape at 100% and NEVER rescale it there. Scaling a
    panel down in Inkscape shrinks its text below the 6 pt floor and desyncs it
    from the rcParams spec -- this is exactly what pushed the draft's axis text to
    ~4.3 pt.

      w_mm, h_mm : final printed size in millimetres. Use the target column width
                   (85 = single, 114 = 1.5-col, 174 = full). Omit to keep the
                   figure's current figsize.

    Writes SVG (text stays editable -- svg.fonttype='none') and PDF (dpi only
    affects raster insets; the plot itself is vector). No bbox_inches='tight':
    that trims to content and lets the outer size drift between exports.
    """
    if w_mm is not None and h_mm is not None:
        fig.set_size_inches(w_mm / 25.4, h_mm / 25.4)
    _out = os.path.join(base_savedir, subdir)
    fig.savefig(os.path.join(_out, name + '.svg'), transparent=True)
    fig.savefig(os.path.join(_out, name + '.pdf'), dpi=600, transparent=True)


## Functions (defined once)

The four analysis functions are byte-identical to the parent. `plot_multiple_errorbars_iqr` has had its embedded styling removed so Figure 2 inherits the central style (its signature is unchanged).

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3


In [ ]:
def plot_multiple_errorbars_iqr(ax, means, std1, std2, c, label, name, yticks):
    """Draw one median line + IQR band on `ax`.

    Visual style (font, ticks, grid, spines) and the y-tick choice are inherited
    from the centralised rcParams + the caller, so this matches the Figure 1 WD
    panel: automatic y-ticks, no forced tick count or 2-decimal formatter.
    `label` and `yticks` are kept in the signature only for backward
    compatibility with the parent notebook's call sites.
    """
    # Median line
    ax.plot(means.index, means, color=c, label=name, linewidth=1.5)  # 1.5 pt = Cell Press max stroke (0.5-1.5 pt)

    # IQR band
    ax.fill_between(means.index, np.asarray(std1), np.asarray(std2),
                    alpha=0.25, edgecolor=c, facecolor=c, linewidth=1)


In [ ]:
def WD_quick(df):
    WD = []
    
    for time in df.index.get_level_values('time').unique():
        row = df.loc[(slice(None), time), 'direction']

        # Ensure row is a NumPy array
        row = np.concatenate(row.values) if isinstance(row.iloc[0], np.ndarray) else row.to_numpy()
        
        if row.size == 0 or np.any(np.isnan(row)):  # Skip if empty or contains NaN
            wd = np.nan
        else:
            row = (row + np.pi) / (2 * np.pi)  # Normalize to [0,1]
            rn = np.random.uniform(0, 1, 10000)
            wd = ot.wasserstein_circle(row, rn,p=1)


        WD.append((time, wd))

    return pd.DataFrame(WD, columns=['time', 'Wasserstein Distance']).set_index('time')


In [ ]:
def process_run_data(df, runs_dict, prefix, suffix_range=50):
    """
    Process the run data for a given DataFrame and store the directions in a dictionary.

    Args:
    df: The DataFrame containing the data.
    runs_dict: The dictionary to store the processed data.
    prefix: The prefix for the run names (e.g., 'run-endocytosis2_baseline_zero_diffusion').
    suffix_range: The range of suffixes to process (default is 50).
    """
    for suffix in range(suffix_range):
        # Generate the run identifier with the correct suffix
        run_to_plot = f'{prefix}{suffix:04d}'  # Format as 0000, 0001, ..., 0049
        
        # Filter the data for the current run
        run_data = df[df['run'] == run_to_plot]
        
        # Check if the filtered run data is empty
        if run_data.empty:
            print(f'{run_to_plot} is empty, skipping.')
            continue  # Skip this run if it's empty

        # Store each direction separately for the current run
        direction_entries = []

        # Loop over unique times in the current run
        for time in run_data['time'].unique():
            time_data = run_data[run_data['time'] == time]
            
            ypos_time = time_data.ypos_recalibrated
            xpos_time = time_data.xpos_recalibrated
            
            # Compute directions (angles)
            directions = np.arctan2(ypos_time, xpos_time)
            
            # Append each direction individually for proper indexing
            for direction in directions:
                direction_entries.append({'run': run_to_plot, 'time': time, 'direction': direction})

        # Convert to DataFrame with MultiIndex for the current run and store in the dictionary
        all_directions = pd.DataFrame(direction_entries).set_index(['run', 'time'])
        
        # Store the DataFrame in the dictionary
        runs_dict[run_to_plot] = all_directions


In [ ]:
def circle_WD(dir_df, loops=1):
    timepoints = dir_df.index.get_level_values('time').unique()
    runs = dir_df.index.get_level_values('run').unique()
    max_length_per_run = dir_df.groupby('run')['length'].max()
    max_length_array = max_length_per_run.values

    all_wass_distances = []

    for time in timepoints:
        for run in runs:
            if (run, time) in dir_df.index:
                row = dir_df.loc[(run, time)]
                length = row['length']
                if length == 0:
                    continue  # Skip short tracks

                directions = row['direction']  # Angular data
                directions_shift = directions

                wass_dists = []
                baseline_sizes = []

                for _ in range(loops):
                    random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                    random_network2 = np.random.uniform(-np.pi, np.pi, len(directions))
                    random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                    random_network2_shift = (np.pi + random_network2) / (2 * np.pi)

                    wass_dist = ot.wasserstein_circle(directions_shift, random_network_shift, p=1)
                    baseline_size = ot.wasserstein_circle(random_network2_shift, random_network_shift, p=1)

                    wass_dists.append(wass_dist)
                    baseline_sizes.append(baseline_size)

                all_wass_distances.append({
                    'run': run,
                    'time': time,
                    'avg_WD': np.mean(wass_dists),
                    'avg_BL': np.mean(baseline_sizes)
                })

    # Create DataFrame
    wass_distances_df = pd.DataFrame(all_wass_distances)

    # Aggregate WD and baseline
    grouped = wass_distances_df.groupby('time').agg({
        'avg_WD': lambda x: median_and_iqr_errors(x.tolist()),
        'avg_BL': lambda x: median_and_iqr_errors(x.tolist())
    })

    # Extract results
    (median_WD, lower_WD, upper_WD) = zip(*grouped['avg_WD'])
    (median_BL, lower_BL, upper_BL) = zip(*grouped['avg_BL'])

    result = pd.DataFrame({
        'Median_WD': median_WD,
        'Lower_WD': lower_WD,
        'Upper_WD': upper_WD,
        'Median_BL': median_BL,
        'Lower_BL': lower_BL,
        'Upper_BL': upper_BL
    }, index=grouped.index)

    # --- Baseline max distribution ---
    last_wd_dfs = []
    for length in max_length_array:
        if length > 0:
            for p_time in range(150):
                rn1 = np.random.uniform(-np.pi, np.pi, length)
                rn2 = np.random.uniform(-np.pi, np.pi, 10000)
                rn1_shift = (np.pi + rn1) / (2 * np.pi)
                rn2_shift = (np.pi + rn2) / (2 * np.pi)
                last_wd = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                last_wd_dfs.append({'time': p_time, 'avg_WD': last_wd})

    baseline_max = pd.DataFrame(last_wd_dfs)

    grouped_baseline = baseline_max.groupby('time').agg({
        'avg_WD': lambda x: median_and_iqr_errors(x.tolist())
    })

    (median_baseline, lower_baseline, upper_baseline) = zip(*grouped_baseline['avg_WD'])

    baseline_max = pd.DataFrame({
        'time': grouped_baseline.index,
        'Median': median_baseline,
        'Lower Error': lower_baseline,
        'Upper Error': upper_baseline
    })

    return result, baseline_max, wass_distances_df


In [ ]:
# Wasserstein distance binned by network size ("amount of actin"). Imported
# byte-identical from the parent (Nonuniformity_manuscript.ipynb) -- calculation
# unchanged. Draws from the same seeded legacy np.random as circle_WD (covered by
# the cell-6 seed) and reuses median_and_iqr_errors defined above.
# loops=1 (canonical); baseline_loops=100.
def circle_size_WD(dir_df, shared_bins, loops=1, baseline_loops=100):
    all_wass_distances = []

    for i in range(len(shared_bins) - 1):
        bin_start, bin_end = shared_bins[i], shared_bins[i + 1]
        subset = dir_df[(dir_df['length'] >= bin_start) & (dir_df['length'] < bin_end)]
        bin_count = len(subset)  # Count of elements in the bin

        if subset.empty:
            continue  # Skip empty bins

        wass_dists = []
        bl_sizes = []

        for _, row in subset.iterrows():
            directions = row['direction']
            #print(row)

            for _ in range(loops):
                random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                random_network2 = np.random.uniform(-np.pi, np.pi, int(round(row['length'])))
                random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                random_network2_shift = (np.pi + random_network2) / (2 * np.pi)

                wass_dist = ot.wasserstein_circle(directions, random_network_shift, p=1)
                bl_size = ot.wasserstein_circle(random_network2_shift, random_network_shift, p=1)

                wass_dists.append(wass_dist)
                bl_sizes.append(bl_size)

        # Compute median and IQR for WD and BL
        median_WD, lower_WD, upper_WD = median_and_iqr_errors(wass_dists)
        median_BL, lower_BL, upper_BL = median_and_iqr_errors(bl_sizes)

        # Baseline estimate (Wasserstein distance between two random uniform networks of size = bin center)
        baseline_wass_dists = []
        for _ in range(baseline_loops):
            baseline_network1 = np.random.uniform(-np.pi, np.pi, 10000)
            baseline_network2 = np.random.uniform(-np.pi, np.pi, int((bin_start + bin_end) / 2))

            baseline_1_shift = (np.pi + baseline_network1) / (2 * np.pi)
            baseline_2_shift = (np.pi + baseline_network2) / (2 * np.pi)

            baseline_wass_dist = ot.wasserstein_circle(baseline_1_shift, baseline_2_shift, p=1)
            baseline_wass_dists.append(baseline_wass_dist)

        baseline_median = np.median(baseline_wass_dists)
        baseline_lower_error = np.percentile(baseline_wass_dists, 25)
        baseline_upper_error = np.percentile(baseline_wass_dists, 75)

        # Append results
        all_wass_distances.append({
            'Bin Center': subset['monomers'].mean(),
            'Median': median_WD,
            'Lower Error': lower_WD,
            'Upper Error': upper_WD,
            'Count': bin_count,
            'Median BL': median_BL,
            'Lower BL': lower_BL,
            'Upper BL': upper_BL,
            'Baseline Median WD': baseline_median,
            'Baseline Lower Error': baseline_lower_error,
            'Baseline Upper Error': baseline_upper_error
        })

    # Convert to DataFrame and return
    result = pd.DataFrame(all_wass_distances)
    result = result.sort_values(by='Bin Center').set_index('Bin Center')

    return result

In [ ]:
def boot_pct_ci(full, other, ci_frac=0.95, reps=10000, seed=0):
    """
    Bootstrap CI of 100*(median(other) - median(full)) / median(full) -- the
    percentage difference in medians relative to the reference. Uses the BCa
    interval (bias- and skew-corrected); if BCa degenerates -- which happens when
    a group has a hard point mass of identical values, e.g. many runs at exactly
    0 monomers -- it falls back to the percentile interval. The ratio is formed
    within each resample, so the interval is bounded >= -100%. `full` must be the
    well-populated reference group (its resampled median is never 0).
    """
    f = np.asarray(full,  float).ravel(); f = f[~np.isnan(f)]
    g = np.asarray(other, float).ravel(); g = g[~np.isnan(g)]

    def _pct(f_s, g_s, axis=-1):
        return 100.0 * (np.median(g_s, axis=axis) - np.median(f_s, axis=axis)) / np.median(f_s, axis=axis)

    def _ci(method):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r = scipy.stats.bootstrap((f, g), _pct, n_resamples=reps, method=method,
                                      vectorized=True, axis=-1,
                                      confidence_level=ci_frac, random_state=seed)
        return r.confidence_interval.low, r.confidence_interval.high

    lo, hi = _ci('BCa')
    if not (np.isfinite(lo) and np.isfinite(hi)):   # BCa degenerated -> percentile
        lo, hi = _ci('percentile')
    return np.array([lo, hi])

In [ ]:
def _ci_label(df, a, b, column):
    lo, hi = boot_pct_ci(df.loc[df['Distribution']==a, column],
                         df.loc[df['Distribution']==b, column], seed=0)
    return f'95% CI [{lo:.1f}%, {hi:.1f}%]'

In [ ]:
def cv_bootstrap(x, ci_frac=0.95, reps=10000, seed=0):
    """Observed CV (std/mean, ddof=1) and its 95% bootstrap CI. Assumes mean > 0."""
    x = np.asarray(x, float).ravel(); x = x[~np.isnan(x)]
    cv = lambda a, axis=-1: np.std(a, axis=axis, ddof=1) / np.mean(a, axis=axis)
    obs = float(cv(x))
    def _ci(method):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r = scipy.stats.bootstrap((x,), cv, n_resamples=reps, method=method,
                                      vectorized=True, axis=-1,
                                      confidence_level=ci_frac, random_state=seed)
        return r.confidence_interval.low, r.confidence_interval.high
    lo, hi = _ci('BCa')
    if not (np.isfinite(lo) and np.isfinite(hi)):      # BCa degenerated -> percentile
        lo, hi = _ci('percentile')
    return obs, lo, hi

In [ ]:
def annotate_below(ax, pairs, order, labels, y_top, dy,
                   leg=0.045, text_gap=0.015, lw=0.6, fs=6, c='0.15'):
    """CI brackets BELOW the data: bar on the bottom, short legs pointing up toward
    the groups, label under the bar. Stacked downward (widest pair lowest)."""
    xi = {c_: i for i, c_ in enumerate(order)}
    for k, ((a, b), lab) in enumerate(zip(pairs, labels)):
        x1, x2 = xi[a], xi[b]
        y = y_top - k * dy
        ax.plot([x1, x1, x2, x2], [y + leg, y, y, y + leg],
                lw=lw, c=c, solid_capstyle='round', clip_on=False)
        ax.text((x1 + x2) / 2, y - text_gap, lab,
                ha='center', va='top', fontsize=fs, color=c, clip_on=False)

# Units: normalization, conversion to radians, and interpretation

**Normalization.** Each branched-actin model point bound to Hip1R is reduced to an angle around the pit centre, $\theta = \mathrm{arctan2}(y, x) \in [-\pi, \pi]$. Before the circular Wasserstein distance (WD) is computed, these angles are rescaled to the unit interval, $\tilde{\theta} = (\theta + \pi)/(2\pi) \in [0, 1]$. This step is required because `ot.wasserstein_circle` measures optimal transport on a circle of **unit circumference**, parameterized by $[0, 1)$.

**Why the raw value is unitless, and why we convert to radians.** Because the metric runs on a circle of circumference 1, the raw distance is a dimensionless *fraction of the circle*, bounded in $[0, 0.5]$. To express it as a physical angle on the original pit circle we undo the normalization by multiplying by $2\pi$ (`WD_TO_RAD = 2*np.pi`). The result is in **radians of arc**, bounded in $[0, \pi]$ — which is why the y-axis is labelled "(rad)". Leaving the `(rad)` label on the unscaled $[0, 0.5]$ values would overstate the quantity by a factor of $2\pi$; scaling by $2\pi$ makes the label correct.

**Interpretation.** The circular WD is the minimum average angular distance the network's angular density must be "transported" to become perfectly uniform around the pit:

- **WD $\approx$ 0 rad** — actin is distributed uniformly around the pit (no angular bias).
- **larger WD** — the network is increasingly non-uniform / one-sided.
- **WD $\to \pi$ rad** — the limiting case of all density at one angle versus its antipode (maximal asymmetry; not reached in practice).

The **simulated uniform distribution** line is the finite-sample baseline: the WD between two independent uniform angular samples of the same size. It marks the floor set by finite point numbers — a simulated network is meaningfully non-uniform only when its WD sits clearly *above* this line.

In [ ]:
# Conversion factor: raw circular WD (unit circle, [0, 0.5]) -> radians ([0, pi]).
# Applied at plot time so the analysis functions (WD_quick, circle_WD) stay
# byte-identical to the parent and keep returning their native [0, 1]-circle WD.
WD_TO_RAD = 2 * np.pi
UNIFORMITY_MAX = np.pi / 2  # WD of a perfectly uniform distribution on the unit circle (radians)
PLOT_UNIFORMITY_METRIC = True


def wd_axis(wd_raw):
    """Raw circular WD -> plotted quantity (radians): WD, or uniformity = pi/2 - WD."""
    x = wd_raw * WD_TO_RAD
    return UNIFORMITY_MAX - x if PLOT_UNIFORMITY_METRIC else x

WD_AXIS_LABEL = 'Uniformity (rad)' if PLOT_UNIFORMITY_METRIC else 'Wasserstein Distance (rad)'

In [ ]:
def set_wd_yaxis(ax):
    ax.set_ylabel(WD_AXIS_LABEL)
    ax.set_ylim([0, UNIFORMITY_MAX])
    if PLOT_UNIFORMITY_METRIC:
        ax.set_yticks(np.append(np.arange(0, UNIFORMITY_MAX, 0.2), UNIFORMITY_MAX))
    else:
        ax.yaxis.set_major_locator(MultipleLocator(0.2))

# Load Data

In [ ]:
# Load the recalibrated points data

arp23 = True
tension = True

if arp23 is True:
    # List of runs
    runs = [
        #'endocytosis2_baseline_aws_output', 
        'endocytosis2_baseline_zero_diffusion_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
        'endocytosis2_baseline_zero_diffusion_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
        #'endocytosis2_baseline_asymmetric_Arp23_34donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_12donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_14donut_aws_output'
    ]

    # Directory containing the output data
    output_dir_base = working_dir + simulations_dir 

    # Dictionaries to store the loaded data
    branched_actin_bound_points = {}
    df_with_lengths = {}

    # Iterate through each run
    for i, run in enumerate(runs):
        try:
            # Construct the output directory for the current run
            output_dir = f"{output_dir_base}{run}"
            
            # Load the recalibrated points data
            branched_actin_bound_points[run] = pd.read_pickle(
                f"{output_dir}/dataframes/branched-actin-bound-to-hip1r_positions_recalibrated.pkl"
            )
            
            # Load and process the dataframe with lengths
            df_with_lengths[run] = pd.read_pickle(
                f"{output_dir}/dataframes/branched-actin-bound-to-hip1r-ends_recalibrated.pkl"
            ).set_index(['run', 'time']).sort_index()
            
            print(f"Arp23 Success, run {i} ({run})")
            
        except Exception as e:
            # Handle any errors encountered during loading
            print(f"Arp23 Failure run {i} ({run}): {e}")

if tension is True:
    # List of runs
    tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

    # Dictionary to store the data for each run
    hip1r_assoc_positions = {}

    # Iterate through each run
    for i, run in enumerate(tension_runs):
        try:
            # Build the file path for the current run's data
            file_path = f'/Volumes/homes/akamatsuadmin/_Cytosim/abhi_simulations/Data_Ben/endocytosis2_baseline_noforcedepcap{run}_aws_output/dataframes/branched-actin-bound-to-hip1r_positions_recalibrated.pkl'
            
            # Load the data and store it in the dictionary
            hip1r_assoc_positions[run] = pd.read_pickle(file_path)
            
            print(f"Tension Success run {i} ({run})")
            
        except Exception as e:
            # Handle errors (e.g., file not found)
            print(f"Tension Failure run {i} ({run}): {e}")


In [ ]:
plot_order = ['_tension10', '', '_tension1000', '_tension5000', '_tension10000']
tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

labels=[10, 150, 1000, 5000, 10000]


# Figure 1 — Polar plots + WD vs time (run0000-0001)

In [ ]:

# Initialize dictionaries
runs_dict_1 = {}
runs_dict_2 = {}
runs_dict_3 = {}

# DataFrames
df1 = branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output']
df2 = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
df3 = branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output']


# Process the two dictionaries
process_run_data(df1, runs_dict_1, 'run-endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut0000-')
process_run_data(df2, runs_dict_2, 'run-endocytosis2_baseline_zero_diffusion0000-')
process_run_data(df3, runs_dict_3, 'run-endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut0000-')


# Check if everything is stored correctly
print(f"Number of runs processed in dict 1: {len(runs_dict_1)}")
print(f"Number of runs processed in dict 2: {len(runs_dict_2)}")
print(f"Number of runs processed in dict 3: {len(runs_dict_3)}")

# # Save the dictionaries to files (e.g., as pickles or CSVs)
# runs_dict_1_df = pd.concat(runs_dict_1.values())
# runs_dict_1_df.to_pickle('runs_dict_1.pkl')  # Save as a pickle file

# runs_dict_2_df = pd.concat(runs_dict_2.values())
# runs_dict_2_df.to_pickle('runs_dict_2.pkl')  # Save as a pickle file

# runs_dict_3_df = pd.concat(runs_dict_3.values())
# runs_dict_3_df.to_pickle('runs_dict_3.pkl')  # Save as a pickle file

# print("Dictionaries have been saved successfully.")


In [ ]:

run_wd_dict_1 = {}
run_wd_dict_2 = {}
run_wd_dict_3 = {}

for run, df in runs_dict_1.items():
    run_wd_dict_1[run] = WD_quick(df)

for run, df in runs_dict_2.items():
    run_wd_dict_2[run] = WD_quick(df)

for run, df in runs_dict_3.items():
    run_wd_dict_3[run] = WD_quick(df)


In [ ]:
# NOTE vs parent: per-call font/grid/tick/spine styling on the WD-vs-time
# panel removed; it now comes from the centralised rcParams. Plot CONTENT
# (colours, geometry, rticks, ylim) is unchanged.


# Timepoints at which to plot polar filament density histograms
timepoints = [2.0, 5.0, 7.0, 13.0, 15.0]
run_to_plot = 'run-endocytosis2_baseline_zero_diffusion0000-0001'
df = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
run_data = df[df['run'] == run_to_plot]
wd_dict_data = run_wd_dict_2[run_to_plot]

N = 5
bottom = 0
theta = np.linspace(-np.pi, np.pi, N, endpoint=True)
theta_mid = [(theta[i] + theta[i+1]) / 2 for i in range(len(theta) - 1)]
width = (2 * np.pi) / N

# Resize figure to be wider than tall for better two-row layout
fig, axs = plt.subplots(2, len(timepoints), figsize=(12, 8), subplot_kw={'polar': True})
fig.suptitle(f'Run {run_to_plot}: Polar Plots')

# Add more space between rows
plt.subplots_adjust(hspace=0.3)

global_max_r = 0

for j, timept in enumerate(timepoints):
    if timept in run_data['time'].values:
        branched_actin_bound_points_recalibrated_time = run_data[run_data['time'] == timept]
        ypos_time = branched_actin_bound_points_recalibrated_time.ypos_recalibrated
        xpos_time = branched_actin_bound_points_recalibrated_time.xpos_recalibrated
        
        ypos_fil = ypos_time * 10**3
        xpos_fil = xpos_time * 10**3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        theta_fil = np.arctan2(ypos_fil.values, xpos_fil.values)
        
        if len(r_fil) > 0:
            global_max_r = max(global_max_r, np.max(r_fil))
        
            direction = np.arctan2(ypos_time, xpos_time)
        else:
            direction = []

    # Row 1: Polar histograms of model points
    ax1 = axs[0, j]
    radii, _ = np.histogram(direction, bins=theta)
    ax1.bar(theta_mid, radii, width=width, bottom=bottom, edgecolor='black', alpha=0.9)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_rticks(np.linspace(0, np.ceil(np.max(radii)), 5, dtype=int))
    ax1.set_title(f'Time {timept} s')
    
    # Reduce margins within each polar plot
    ax1.set_rorigin(-2.5)
    
    # Row 2: Radial plots of filament points
    ax2 = axs[1, j]
    ax2.scatter(theta_fil, r_fil, c='blue', s=10, alpha=0.7, label='Filament Points')
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    ax2.set_title(f'Filament Points at {timept} s')
    
    # Reduce margins within each polar plot
    ax2.set_rorigin(-2.5)

for j in range(len(timepoints)):
    ax2 = axs[1, j]
    ax2.set_ylim([0, global_max_r+10])
    ax2.set_rticks(np.linspace(0, global_max_r+10, 5, dtype=int))

# Third plot (Wasserstein Distance) - Non-polar plot
fig2, axes2 = plt.subplots(1, 1, figsize=(85 / 25.4, 50 / 25.4))  # final ~85 x 50 mm (single column)
axes2.plot(wd_dict_data.index, wd_axis(wd_dict_data['Wasserstein Distance']), color='black', label= WD_AXIS_LABEL)
# --- Dashed reference markers at t = 4 s and 15 s, drawn only up to the WD curve ---
# Cell Press style: 0.5 pt stroke (lightest weight), dashed, grey, beneath the data line.
_t_arr  = np.asarray(wd_dict_data.index, dtype=float)
_wd_arr = np.asarray(wd_axis(wd_dict_data['Wasserstein Distance']), dtype=float)
_o = np.argsort(_t_arr); _t_arr, _wd_arr = _t_arr[_o], _wd_arr[_o]
for _t in (4.0, 10.0, 15.0):
    _y = np.interp(_t, _t_arr, _wd_arr)          # height where the marker meets the curve
    axes2.plot([_t, _t], [0.0, _y], linestyle=(0, (4, 2)),
               linewidth=0.5, color='0.3', zorder=1)

axes2.set_xlabel('Time (s)')
axes2.set_xticks(list(range(0, 15, 2)) + [15])  # usual 0–14 by 2, plus 15
set_wd_yaxis(axes2)

# Use tight_layout with adjusted parameters
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave room for the suptitle
# save_panel(fig2, 'wasserstein_distance_time', 'figure1', w_mm=85, h_mm=50)  # export at final size; place 1:1 in Inkscape
plt.show()


# Figure 1 (cont.) — WD vs time across Arp2/3 distributions


In [ ]:
# ---------------------------------------------------------------------------
# Figure 1 configuration -- which initial Arp2/3 distributions to include.
#
# Flip `show` to add/remove a distribution. This single config is the source of
# truth: it drives BOTH the computation cell and the plot cell below, and it
# supersedes the zero_trials list from the "Load Data" section.
#
# v1 of the figure: only the uniform Arp2/3 case (legend 'Simulated actin networks') plus the Uniform
# random baseline. Switch the donuts on later by setting their `show` to True.
#
# NOTE: `show` gates COMPUTATION as well as plotting (circle_WD is slow, so we
# don't compute distributions we won't draw). Turning one on therefore means
# re-running the compute cell, then the plot cell. Colours are assigned over
# the full ordered set, so each distribution keeps a stable colour regardless
# of which subset is shown.
# ---------------------------------------------------------------------------
import seaborn as sns

arp23_distributions = {
    # output key                                                          : config
    'endocytosis2_baseline_zero_diffusion_output':                          {'label': 'Simulated branched-actin networks', 'show': True},
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output': {'label': '3/4 ring',  'show': True},
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output': {'label': '1/2 ring',  'show': True},
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output': {'label': '1/4 ring',  'show': True},
}

# Show the black "Uniform" random-reference line?
show_uniform_baseline = True

# Stable colour per distribution (viridis over the full ordered set).
_all_keys = list(arp23_distributions.keys())
_palette = sns.color_palette("viridis", len(_all_keys))
for _i, _k in enumerate(_all_keys):
    arp23_distributions[_k]['color'] = _palette[_i]

# Selected distributions -> drives both compute and plot.
zero_trials = [k for k, v in arp23_distributions.items() if v['show']]
if not zero_trials:
    raise ValueError('No Arp2/3 distributions selected (set at least one show=True).')

# Source trial for the Uniform baseline line (prefer the uniform/Full case).
_baseline_src = 'endocytosis2_baseline_zero_diffusion_output'
if _baseline_src not in zero_trials:
    _baseline_src = zero_trials[0]

print('Figure 1 will include:',
      [arp23_distributions[k]['label'] for k in zero_trials],
      '| baseline:', show_uniform_baseline)


In [ ]:
# zero_trials is supplied by the Figure 1 config cell above (flag-controlled);
# only the selected distributions are computed here.

WD_zero_dict = {}
baseline_dict = {}

ypos_recalibrated = []
xpos_recalibrated = []
direction = []
lengths = []
index = np.round(np.arange(0, 15.0, 0.1), 1) 

# Recalibrate positions and calculate directions
for trial in zero_trials:
    if trial in branched_actin_bound_points:
        ypos_recalibrated.append(branched_actin_bound_points[trial].dropna().ypos_recalibrated)
        xpos_recalibrated.append(branched_actin_bound_points[trial].dropna().xpos_recalibrated)
        dir = np.arctan2(ypos_recalibrated[-1], xpos_recalibrated[-1])
        dir_circle = (np.pi + dir) / (2 * np.pi)  # Convert to [0, 1] for circular WD
        direction.append(dir_circle)
        
        # Ensure NaN values are considered as length zero
        lengths.append(np.sum(~np.isnan(dir)))  # Count non-NaN values for length
        branched_actin_bound_points[trial]['direction'] = dir_circle

# Group directions by 'run' and 'time' and add lengths as a column
directions_grouped_dict = {}
per_run_WD = {}

for trial in zero_trials:
    # Group the directions
    grouped_directions = branched_actin_bound_points[trial].groupby(['run', 'time']).direction.apply(np.array)
    
    # Create a DataFrame with 'direction' and 'length' columns
    grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))  # Count non-NaN values
    directions_grouped_dict[trial] = pd.DataFrame({
        'direction': grouped_directions,
        'length': grouped_lengths
    })
    # Get DF for current trial
    dir_df = directions_grouped_dict[trial]

    # Calculate corrected WD
    result, baseline, per_run = circle_WD(dir_df)
    result.index = index[:len(result)]
    baseline.index = index[:len(result)]
 
    WD_zero_dict[trial] = result
    baseline_dict[trial] = baseline
    per_run_WD[trial] = per_run

    print(f"Successfully processed trial: {trial}")


In [ ]:
# Figure 1 (cont.) -- WD vs time for the selected Arp2/3 distributions.
# All visual style comes from the centralised rcParams (matches the Fig 1 WD
# panel): Arial labels, no grid, top/right spines off, outward ticks, and
# AUTOMATIC y-ticks (no forced count / formatter) so the ticks match Fig 1.
# plot_multiple_errorbars_iqr() only draws data; decoration is done once.
# WD values are converted to radians (x 2*pi = WD_TO_RAD) -- see the units note.

# --- Y-axis range from whatever is currently shown (in radians) ---
all_max = [np.max(WD_zero_dict[t]['Upper_WD']) for t in zero_trials if t in WD_zero_dict]
if show_uniform_baseline and _baseline_src in baseline_dict:
    all_max.append(np.max(baseline_dict[_baseline_src]['Upper Error']))
max_wd = max(all_max) * WD_TO_RAD

# --- Plot ---
fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))   # final ~168 x 74 mm (full-width bottom row per Fig 1 layout template; E is the key panel)

for trial in zero_trials:
    if trial not in WD_zero_dict:
        continue
    if trial != zero_trials[0]:
        continue  # Only plot the first distribution for now (the uniform case)
    d = arp23_distributions[trial]
    plot_multiple_errorbars_iqr(
        ax,
        wd_axis(WD_zero_dict[trial]['Median_WD']),
        wd_axis(WD_zero_dict[trial]['Lower_WD']),
        wd_axis(WD_zero_dict[trial]['Upper_WD']),
        d['color'], 'Wasserstein Distance', d['label'], yticks=None,
    )

# # Uniform random baseline (black)
# if show_uniform_baseline and _baseline_src in WD_zero_dict:
#     plot_multiple_errorbars_iqr(
#         ax,
#         wd_axis(WD_zero_dict[_baseline_src]['Median_BL']),
#         wd_axis(WD_zero_dict[_baseline_src]['Lower_BL']),
#         wd_axis(WD_zero_dict[_baseline_src]['Upper_BL']),
#         'black', 'Wasserstein Distance', 'Simulated uniform distribution (size-matched)', yticks=None,
#     )

# --- Axis decoration (once; style from rcParams; automatic y-ticks like Fig 1) ---
ax.set_xlabel('Time (s)')
ax.set_xticks(list(range(0, 15, 2)) + [15])
set_wd_yaxis(ax)
# ax.legend()
# ax.set_title('Wasserstein Distance Over Time for Different Arp2/3 Distributions')  # off to match Fig 1
plt.tight_layout()
# save_panel(fig, 'wasserstein_distance_time_Arp23distributions', 'figure1', w_mm=85, h_mm=50)  # export at final size; place 1:1 in Inkscape
plt.show()


# Figure 2

In [ ]:
# --- Y-axis range from whatever is currently shown (in radians) ---
all_max = [np.max(WD_zero_dict[t]['Upper_WD']) for t in zero_trials if t in WD_zero_dict]
if show_uniform_baseline and _baseline_src in baseline_dict:
    all_max.append(np.max(baseline_dict[_baseline_src]['Upper Error']))
max_wd = max(all_max) * WD_TO_RAD

# --- Plot ---
fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))   # final ~168 x 74 mm (full-width bottom row per Fig 1 layout template; E is the key panel)

for trial in zero_trials:
    if trial not in WD_zero_dict:
        continue
    d = arp23_distributions[trial]
    plot_multiple_errorbars_iqr(
        ax,
        wd_axis(WD_zero_dict[trial]['Median_WD']),
        wd_axis(WD_zero_dict[trial]['Lower_WD']),
        wd_axis(WD_zero_dict[trial]['Upper_WD']),
        d['color'], 'Wasserstein Distance', d['label'], yticks=None,
    )

# # Uniform random baseline (black)
# if show_uniform_baseline and _baseline_src in WD_zero_dict:
#     plot_multiple_errorbars_iqr(
#         ax,
#         wd_axis(WD_zero_dict[_baseline_src]['Median_BL']),
#         wd_axis(WD_zero_dict[_baseline_src]['Lower_BL']),
#         wd_axis(WD_zero_dict[_baseline_src]['Upper_BL']),
#         'black', 'Wasserstein Distance', 'Simulated uniform distribution (size-matched)', yticks=None,
#     )

# --- Axis decoration (once; style from rcParams; automatic y-ticks like Fig 1) ---
ax.set_xlabel('Time (s)')
ax.set_xticks(list(range(0, 15, 2)) + [15])
set_wd_yaxis(ax)
ax.legend()
# ax.set_title('Wasserstein Distance Over Time for Different Arp2/3 Distributions')  # off to match Fig 1
plt.tight_layout()
# save_panel(fig, 'wasserstein_distance_time_Arp23distributions', 'figure2', w_mm=85, h_mm=50)  # export at final size; place 1:1 in Inkscape
plt.show()


In [ ]:
# Figure 2 -- WD at the last simulated timepoint, per distribution (box + strip).
# WD values come straight from circle_WD's per-run output (per_run_WD), so they
# are byte-identical to the WD-vs-time panel; no WD is recomputed here.
from matplotlib.ticker import MultipleLocator

_short = lambda t: 'Full' if t == _baseline_src else arp23_distributions[t]['label']

_rows = []
for trial in zero_trials:
    d = per_run_WD[trial]
    last = d[d['time'] == d['time'].max()]
    for _, r in last.iterrows():
        _rows.append({'Distribution': _short(trial), 'WD_rad': wd_axis(r['avg_WD'])})
wd_last_df = pd.DataFrame(_rows)

order = [_short(t) for t in zero_trials]
pal   = [arp23_distributions[t]['color'] for t in zero_trials]

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
sns.boxplot(data=wd_last_df, x='Distribution', y='WD_rad', order=order,
            hue='Distribution', hue_order=order, palette=pal, legend=False,
            showfliers=False, linewidth=0.6, width=0.6,
            boxprops=dict(alpha=0.35), ax=ax)
sns.stripplot(data=wd_last_df, x='Distribution', y='WD_rad', order=order,
              hue='Distribution', hue_order=order, palette=pal, legend=False,
              size=2, jitter=0.2, alpha=0.9, ax=ax)
ax.set_xlabel('Initial Arp2/3 distribution')
set_wd_yaxis(ax)


pairs = [('Full', '3/4 ring'), ('Full', '1/2 ring'), ('Full', '1/4 ring')]

# labels = [_ci_label(wd_last_df, a, b, 'WD_rad') for a, b in pairs]
# annotate_below(ax, pairs, order, labels, y_top=0.72, dy=0.17)

annot = Annotator(ax, pairs, data=wd_last_df, x='Distribution', y='WD_rad', order=order)
annot.configure(loc='outside', fontsize=6, line_width=0.6, verbose=0)
annot.set_custom_annotations([_ci_label(wd_last_df, a, b, 'WD_rad') for a, b in pairs])
annot.annotate()

plt.tight_layout()
# save_panel(fig, 'wasserstein_distance_last_timepoint', 'figure2', w_mm=85, h_mm=50)
plt.show()

In [ ]:
# Figure 2 (compute) -- Wasserstein distance vs network size ("amount of actin").
# Reuses directions_grouped_dict from the Figure 1 (cont.) compute cell. That dict
# is grouped from the FULL frame, so it also contains empty groups (length == 0,
# e.g. t=0 before any actin binds); the parent's size-analysis grouping dropped
# those upstream via .dropna(). circle_WD tolerates them (internal `length == 0`
# skip) but circle_size_WD does not, so we filter to length > 0 here --
# reproducing exactly the (non-empty) groups the parent fed the function.
# Binning is identical to the parent: 70 fixed-width length bins per trial.
WD_zero_size_dict = {}

for trial in zero_trials:
    if trial not in directions_grouped_dict:
        continue

    dir_df = directions_grouped_dict[trial]
    dir_df = dir_df[dir_df['length'] > 0]   # drop empty groups (parent dropped these upstream via .dropna())

    mono = df_with_lengths[trial].groupby(level=['run', 'time'])['length'].sum() * (1000 / 2.75)
    dir_df = dir_df.join(mono.rename('monomers'))
    n_missing = int(dir_df['monomers'].isna().sum())
    if n_missing:
        print(f"  WARNING: {n_missing} snapshot(s) in positions have no ends match (monomers=NaN)")


    # Fixed-width length bins from this trial's min to max length.
    trial_lengths = dir_df['length'].dropna().values
    n_bins = 151
    bin_edges = np.linspace(trial_lengths.min(), trial_lengths.max(), n_bins + 1)

    WD_zero_size_dict[trial] = circle_size_WD(dir_df, bin_edges)
    print(f"Successfully processed trial (WD vs size): {trial}")

In [ ]:
# Figure 1 panel -- plotted here in the Figure 2 section for convenience, because it
# reuses the Figure 2 size dataframe (WD_zero_size_dict). Uniformity vs amount of
# actin for the full-ring branched-actin network, with the size-matched uniform
# baseline. Per-bin values come straight from circle_size_WD (no recompute) and are
# mapped to the plotted axis by wd_axis (uniformity = pi/2 - WD in radians when
# PLOT_UNIFORMITY_METRIC is on, WD in radians otherwise); style from the central
# rcParams. Only the full ring + uniform baseline are shown; the x-axis is actin
# amount, so the time-axis tick convention does not apply. y-tick increment 0.2 rad.
from matplotlib.ticker import MultipleLocator

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))

# Full ring == zero_trials[0] == _baseline_src; its size-matched baseline is drawn too.
r = WD_zero_size_dict[_baseline_src]

plot_multiple_errorbars_iqr(
    ax,
    wd_axis(r['Median']),
    wd_axis(r['Lower Error']),
    wd_axis(r['Upper Error']),
    arp23_distributions[_baseline_src]['color'], 'Wasserstein Distance',
    'Simulated branched-actin networks', yticks=None,
)

# Uniform baseline (black).
plot_multiple_errorbars_iqr(
    ax,
    wd_axis(r['Median BL']),
    wd_axis(r['Lower BL']),
    wd_axis(r['Upper BL']),
    'black', 'Wasserstein Distance', 'Simulated uniform distribution', yticks=None,
)

ax.set_xlabel('Actin amount (monomers)')
ax.set_xticks(np.arange(0, 5001, 1000))
ax.set_xlim([0, 5000])
ax.set_ylabel(WD_AXIS_LABEL)

set_wd_yaxis(ax)

ax.legend(loc='lower right')
plt.tight_layout()
# save_panel(fig, 'wasserstein_distance_vs_actin_full_ring', 'figure1', w_mm=85, h_mm=50)  # export at final size; place 1:1 in Inkscape
plt.show()

In [ ]:
# Figure 2 -- Wasserstein distance vs amount of actin, per distribution
# (median + IQR band) plus the size-matched uniform baseline. WD per bin comes
# straight from circle_size_WD (no recompute); scaled to radians (x WD_TO_RAD)
# and styled from the centralised rcParams, matching the WD-vs-time panels.
# y-tick increment 0.2 rad (limit free -- constrained cases exceed 1.4); the
# x-axis is actin amount, so the time-axis tick convention does not apply.
from matplotlib.ticker import MultipleLocator

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))

baseline_plotted = False
for trial in zero_trials:
    if trial not in WD_zero_size_dict:
        continue
    d = arp23_distributions[trial]
    r = WD_zero_size_dict[trial]

    plot_multiple_errorbars_iqr(
        ax,
        wd_axis(r['Median']),
        wd_axis(r['Lower Error']),
        wd_axis(r['Upper Error']),
        d['color'], WD_AXIS_LABEL, d['label'], yticks=None,
    )

    # # Size-matched uniform baseline (black), drawn once (from the first distribution's bins, as in the parent).
    # if not baseline_plotted:
    #     plot_multiple_errorbars_iqr(
    #         ax,
    #         wd_axis(r['Median BL']),
    #         wd_axis(r['Lower BL']),
    #         wd_axis(r['Upper BL']),
    #         'black', WD_AXIS_LABEL, 'Simulated uniform distribution (size-matched)', yticks=None,
    #     )
    #     baseline_plotted = True

ax.set_xlabel('Actin amount (monomers)')
ax.set_xticks(np.arange(0, 7001, 1000))
ax.set_xlim([0, 7000])
set_wd_yaxis(ax)
ax.legend()
plt.tight_layout()
# save_panel(fig, 'wasserstein_distance_vs_actin', 'figure2', w_mm=85, h_mm=50)  # export at final size; place 1:1 in Inkscape
plt.show()

In [ ]:
# Figure 2 (compute) -- network composition per run at the last timepoint, from
# df_with_lengths (one row per fiber per (run, time); a run with no fibers at a
# timepoint is a single all-NaN row). Per the spec those NaNs count as 0
# (0 filaments / 0 length) and every run is retained via reindex, so the
# distributions aren't biased by silently dropping empty runs.
#   Filaments = non-NaN fiber_id count per run
#   Monomers  = sum of fiber length (NaN->0) * 1000/2.75  (2.75 nm rise per monomer)
MONOMERS_PER_UM = 1000 / 2.75

_short = lambda t: 'Full' if t == _baseline_src else arp23_distributions[t]['label']
order  = [_short(t) for t in zero_trials]
pal    = [arp23_distributions[t]['color'] for t in zero_trials]

_frames = []
for trial in zero_trials:
    df = df_with_lengths[trial]
    t_last = df.index.get_level_values('time').max()                 # 15.0
    last = df[df.index.get_level_values('time') == t_last]
    all_runs = df.index.get_level_values('run').unique()
    fil = last.groupby(level='run')['fiber_id'].count().reindex(all_runs, fill_value=0)
    mon = (last.groupby(level='run')['length'].sum() * MONOMERS_PER_UM).reindex(all_runs, fill_value=0)
    _frames.append(pd.DataFrame({'Distribution': _short(trial),
                                 'Filaments': fil.values,
                                 'Monomers':  mon.values}))
network_last_df = pd.concat(_frames, ignore_index=True)

In [ ]:
# Figure 2 -- filament count per run at the last timepoint (box + strip).
fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
sns.boxplot(data=network_last_df, x='Distribution', y='Filaments', order=order,
            hue='Distribution', hue_order=order, palette=pal, legend=False,
            showfliers=False, linewidth=0.6, width=0.6,
            boxprops=dict(alpha=0.35), ax=ax)
sns.stripplot(data=network_last_df, x='Distribution', y='Filaments', order=order,
              hue='Distribution', hue_order=order, palette=pal, legend=False,
              size=2, jitter=0.2, alpha=0.9, ax=ax)
ax.set_xlabel('Initial Arp2/3 distribution')
ax.set_ylabel('Number of actin filaments')
ax.set_ylim(bottom=0)
plt.tight_layout()
# save_panel(fig, 'filaments_last_timepoint', 'figure2', w_mm=85, h_mm=50)
plt.show()

In [ ]:
# Figure 2 -- monomer count per run at the last timepoint (box + strip).
# monomers = total fiber length * 1000/2.75.
fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
sns.boxplot(data=network_last_df, x='Distribution', y='Monomers', order=order,
            hue='Distribution', hue_order=order, palette=pal, legend=False,
            showfliers=False, linewidth=0.6, width=0.6,
            boxprops=dict(alpha=0.35), ax=ax)
sns.stripplot(data=network_last_df, x='Distribution', y='Monomers', order=order,
              hue='Distribution', hue_order=order, palette=pal, legend=False,
              size=2, jitter=0.2, alpha=0.9, ax=ax)
ax.set_xlabel('Initial Arp2/3 distribution')
ax.set_ylabel('Actin amount (monomers)')
ax.set_ylim(bottom=0)

# 95% CI of the median difference (donut - Full) per pair, bootstrap; rendered
# through statannotations so the bracket/label styling matches the other panels.
pairs = [('Full', '3/4 ring'), ('Full', '1/2 ring'), ('Full', '1/4 ring')]

annot = Annotator(ax, pairs, data=network_last_df, x='Distribution', y='Monomers', order=order)
annot.configure(loc='inside', fontsize=6, line_width=0.6, verbose=0)
annot.set_custom_annotations([_ci_label(network_last_df, a, b, 'Monomers') for a, b in pairs])
annot.annotate()

# # Mann-Whitney test
# annot = Annotator(ax, pairs, data=network_last_df, x='Distribution', y='Monomers', order=order)
# annot.configure(test='Mann-Whitney', loc='inside', fontsize=6, line_width=0.6, verbose=0)
# annot.apply_test()
# _fmt = lambda p: 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
# annot.set_custom_annotations([_fmt(res.data.pvalue) for res in annot.annotations])
# annot.annotate()

plt.tight_layout()
# save_panel(fig, 'monomers_last_timepoint', 'figure2', w_mm=85, h_mm=50)
plt.show()

In [ ]:
# Figure 2 -- coefficient of variation (CV = std/mean) of monomer count per run
# at the last timepoint, one value per Arp2/3 condition. Marker = observed CV;
# error bars = 95% CI from 10000 bootstrap replicates (BCa, percentile fallback
# if BCa degenerates). CV uses the sample std (ddof=1); multiply by 100 for %.
cv_rows = []
for cond in order:
    x = network_last_df.loc[network_last_df['Distribution'] == cond, 'Monomers']
    cv, lo, hi = cv_bootstrap(x, reps=10000, seed=0)
    cv_rows.append({'Distribution': cond, 'n': int(x.notna().sum()),
                    'CV': cv, 'ci_low': lo, 'ci_high': hi})
    print(f"{cond:10s}  n={int(x.notna().sum()):3d}   CV = {cv:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
cv_df = pd.DataFrame(cv_rows)

# CV per condition with asymmetric 95% CI error bars, matching the panel aesthetic
fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
xpos = np.arange(len(order))
yerr = np.vstack([cv_df['CV'] - cv_df['ci_low'], cv_df['ci_high'] - cv_df['CV']])
for i, cond in enumerate(order):
    ax.errorbar(xpos[i], cv_df.loc[i, 'CV'], yerr=yerr[:, i:i+1], fmt='o', ms=4,
                color=pal[i], ecolor=pal[i], elinewidth=0.6, capsize=2, capthick=0.6)
ax.set_xticks(xpos); ax.set_xticklabels(order)
ax.set_xlabel('Initial Arp2/3 distribution')
ax.set_ylabel('Coefficient of variation\n (number of actin monomers)')
ax.set_ylim(bottom=0); ax.set_xlim(-0.5, len(order) - 0.5)
plt.tight_layout()
# save_panel(fig, 'cv_last_timepoint', 'figure2', w_mm=85, h_mm=50)
plt.show()

In [ ]:
# Figure 2 -- monomer count per run at the last timepoint, EXCLUDING empty runs.
# Same panel as the monomer box+strip above, but runs with no fibers at t=15.0
# (the all-NaN / 0-filament cases) are dropped rather than counted as 0, so the
# median/box/whiskers/strip all reflect only runs that actually built a network.
mono_nonempty = network_last_df[network_last_df['Filaments'] > 0]

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
sns.boxplot(data=mono_nonempty, x='Distribution', y='Monomers', order=order,
            hue='Distribution', hue_order=order, palette=pal, legend=False,
            showfliers=False, linewidth=0.6, width=0.6,
            boxprops=dict(alpha=0.35), ax=ax)
sns.stripplot(data=mono_nonempty, x='Distribution', y='Monomers', order=order,
              hue='Distribution', hue_order=order, palette=pal, legend=False,
              size=2, jitter=0.2, alpha=0.9, ax=ax)
ax.set_xlabel('Initial Arp2/3 distribution')
ax.set_ylabel('Number of actin monomers (non-empty)')
ax.set_ylim(bottom=0)

pairs = [('Full', '3/4 ring'), ('Full', '1/2 ring'), ('Full', '1/4 ring')]

# annot = Annotator(ax, pairs, data=mono_nonempty, x='Distribution', y='Monomers', order=order)
# annot.configure(test='Mann-Whitney', loc='inside', fontsize=6, line_width=0.6, verbose=0)
# annot.apply_test()
# _fmt = lambda p: 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
# annot.set_custom_annotations([_fmt(res.data.pvalue) for res in annot.annotations])
# annot.annotate()

annot = Annotator(ax, pairs, data=mono_nonempty, x='Distribution', y='Monomers', order=order)
annot.configure(loc='inside', fontsize=6, line_width=0.6, verbose=0)
annot.set_custom_annotations([_ci_label(mono_nonempty, a, b, 'Monomers') for a, b in pairs])
annot.annotate()

plt.tight_layout()
# save_panel(fig, 'monomers_last_timepoint_nonempty', 'figure2', w_mm=85, h_mm=50)
plt.show()

In [ ]:
# Figure 2 -- bud internalization per run at the last timepoint (box + strip), in nm.
# bud_internalization is recorded on EVERY snapshot in df_with_lengths (including
# the no-fiber placeholder rows) and is constant within a (run, time), so the
# per-run value is just that snapshot's value and every run is represented without
# any NaN->0 or exclusion step. (This is what disambiguates the monomer panel's
# empty runs: a completed event sits high here, a stalled one sits near 0.)
# Values are converted um -> nm (x1000).
_short = lambda t: 'Full' if t == _baseline_src else arp23_distributions[t]['label']
order  = [_short(t) for t in zero_trials]
pal    = [arp23_distributions[t]['color'] for t in zero_trials]

_frames = []
for trial in zero_trials:
    df = df_with_lengths[trial]
    t_last = df.index.get_level_values('time').max()                       # 15.0
    last = df[df.index.get_level_values('time') == t_last]
    intern = last.groupby(level='run')['bud_internalization'].mean().dropna() * 1000   # um -> nm
    _frames.append(pd.DataFrame({'Distribution': _short(trial),
                                 'Internalization': intern.values}))
intern_last_df = pd.concat(_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(85 / 25.4, 50 / 25.4))
sns.boxplot(data=intern_last_df, x='Distribution', y='Internalization', order=order,
            hue='Distribution', hue_order=order, palette=pal, legend=False,
            showfliers=False, linewidth=0.6, width=0.6,
            boxprops=dict(alpha=0.35), ax=ax)
sns.stripplot(data=intern_last_df, x='Distribution', y='Internalization', order=order,
              hue='Distribution', hue_order=order, palette=pal, legend=False,
              size=2, jitter=0.2, alpha=0.9, ax=ax)
ax.set_xlabel('Initial Arp2/3 distribution')
ax.set_ylabel('Bud internalization (nm)')
# y-limits left free: internalization can sit slightly below 0 (bud ~ baseline),
# so we don't clamp bottom=0 the way the WD / monomer panels do.
plt.tight_layout()
# save_panel(fig, 'bud_internalization_last_timepoint', 'figure2', w_mm=85, h_mm=50)
plt.show()